# PrecisionMiner Tutorial

This tutorial demonstrates how to use the PrecisionMiner components of the EpiScope package.  PrecisionMiner performs deep parsing and structured extraction on individual papers.  In this example we use the `LLMExtractor` to extract data source information from a set of relevant text chunks.  We also illustrate how to classify a chunk according to paper type using a simple similarity approach.  To keep the example self‑contained, heavy dependencies such as FAISS and large language models are patched out.

## Extract Data Sources with LLMExtractor

The `LLMExtractor` class wraps an LLM chat function and parses the result into structured data.  We supply a dummy chat function that returns a fixed JSON string.  The extractor then validates the structure using pydantic models and produces an `ExtractionResult`.


In [ ]:
from episcope.parse.extraction.llm_extractor import LLMExtractor
import json

# Define a dummy chat function that returns a JSON object matching the schema.
# We use json.dumps to build the JSON string to avoid escaping issues.
def dummy_chat_fn(model_name: str, messages: list, options: dict) -> dict:
    payload = {
        'data_sources_description': 'Clinical trial database',
        'data_sources': [{
            'source_name': 'ClinicalTrials.gov',
            'url': 'https://clinicaltrials.gov',
            'explanation': 'Repository of clinical trial records',
            'section_found': 'Methods'
        }],
        'references': []
    }
    return {'content': json.dumps(payload)}

# Create an extractor with the dummy chat function
extractor = LLMExtractor(chat_fn=dummy_chat_fn)

# Minimal metadata object with title and abstract
class Meta:
    title = 'Sample Paper'
    abstract = 'This study investigates the efficacy of vaccination.'

# Define relevant text chunks (could be from a PDF)
chunks = [
    {'text': 'The study recruited 100 participants from multiple clinics.'},
    {'text': 'Data were collected from the ClinicalTrials.gov registry.'}
]

# Extract data sources
extraction_result, _ = extractor.extract_data_sources(
    relevant_chunks=chunks,
    references=[],
    paper_type='data_analysis',
    metadata=Meta(),
    query='What data sources were used in this study?'
)

# Display the result
from pprint import pprint
pprint(extraction_result)

## Classify a Chunk by Paper Type

The `PaperClassifier` uses semantic similarity to categorize chunks into paper types (e.g., literature review, data analysis).  For this example we patch the sentence transformer with a dummy implementation that returns a vector whose value is the number of words.  We then call the internal `_classify_chunk_by_templates` method to determine the paper type.


In [ ]:
# Patch the SentenceTransformer within PaperClassifier to avoid loading models
from unittest.mock import patch
import numpy as np
from episcope.parse.extraction.paper_classifier import PaperClassifier

class DummySentenceTransformer:
    def __init__(self, *args, **kwargs):
        pass
    def encode(self, texts, **kwargs):
        if isinstance(texts, list):
            return np.array([[len(t.split())] for t in texts])
        return np.array([len(texts.split())])

with patch('episcope.parse.extraction.paper_classifier.SentenceTransformer', DummySentenceTransformer):
    classifier = PaperClassifier(model_name='dummy', embedding_model='dummy', use_gpu=False)
    text = 'We analysed data from the national survey including thousands of participants.'
    paper_type = classifier._classify_chunk_by_templates(text)
    print(f'Chunk: {text}
Classified as: {paper_type}')

## Summary

This notebook illustrated how to use key components of the PrecisionMiner pipeline without relying on external services.  We extracted structured data sources from simple text chunks and performed a basic paper type classification using a dummy embedding model.  In a full pipeline you would run the `parse/processing/pipeline.py` script to orchestrate GROBID parsing, OCR, table extraction, and the LLM extractor, saving the results in your project directory.